<a href="https://colab.research.google.com/github/islam37/S-SLTE-Thesis/blob/main/notebooks/04_pilot_error_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import json
import hashlib
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

# 1. Load frozen experiment metadata
with open(
    BASE / "configs/llm_pilot_manifest_v2.json",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

revision = manifest["model_revision_sha"]

# 2. Verify the original pilot
pilot_path = BASE / "data/pilot/pilot_500.csv"

assert hashlib.sha256(
    pilot_path.read_bytes()
).hexdigest() == manifest["pilot_sha256"]

pilot = pd.read_csv(pilot_path)

assert len(pilot) == 500
assert pilot["record_id"].is_unique
assert "label" not in pilot.columns

# 3. Load frozen LLM labels
annotation_path = (
    BASE / "outputs/llm_pilot"
    / f"pinned_{revision[:12]}"
    / "pilot_annotations.csv"
)

annotations = pd.read_csv(annotation_path)

assert len(annotations) == 500
assert annotations["record_id"].is_unique
assert annotations["is_valid"].eq(True).all()
assert annotations["llm_label"].isin(range(4)).all()

# 4. Load PRIVATE reference labels
# Strictly for retrospective analysis
reference = pd.read_csv(
    BASE / "private_reference/noisy_reference.csv"
)

assert reference["record_id"].is_unique
assert reference["label"].isin(range(4)).all()

# Ensure all records correspond to the fixed pilot
text_check = pilot.merge(
    reference[["record_id", "text"]],
    on="record_id",
    how="left",
    validate="one_to_one",
    suffixes=("_pilot", "_reference")
)

assert len(text_check) == 500
assert (
    text_check["text_pilot"] ==
    text_check["text_reference"]
).all()

# 5. Construct retrospective evaluation table
df = annotations[["record_id", "llm_label"]].merge(
    reference[["record_id", "label"]],
    on="record_id",
    how="left",
    validate="one_to_one"
)

assert len(df) == 500
assert df["label"].notna().all()
assert set(df["record_id"]) == set(pilot["record_id"])

LABEL_MAP = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# 6. Calculate class transition counts
confusion = pd.crosstab(
    df["label"],
    df["llm_label"]
).reindex(
    index=range(4),
    columns=range(4),
    fill_value=0
)

# Convert each row into transition percentages
row_totals = confusion.sum(axis=1)

transition_pct = (
    confusion.div(row_totals, axis=0) * 100
)

transition_pct.index = [
    LABEL_MAP[i] for i in range(4)
]
transition_pct.columns = [
    LABEL_MAP[i] for i in range(4)
]

print("CLASS TRANSITION MATRIX (%)")
print("Rows: Reference Labels")
print("Columns: LLM Predictions")
display(transition_pct.round(2))

# 7. Per-class disagreement statistics
class_stats = []

for label in range(4):
    total = int(row_totals.loc[label])
    correct = int(confusion.loc[label, label])
    disagreements = total - correct

    class_stats.append({
        "category": LABEL_MAP[label],
        "total": total,
        "matches": correct,
        "disagreements": disagreements,
        "disagreement_pct": round(
            disagreements / total * 100, 2
        )
    })

class_df = pd.DataFrame(class_stats)

print("\nCLASS-WISE DISAGREEMENT")
display(class_df)

# 8. Identify the largest directional transitions
transitions = []

for actual in range(4):
    for predicted in range(4):

        if actual == predicted:
            continue

        count = int(
            confusion.loc[actual, predicted]
        )

        transitions.append({
            "reference_class": LABEL_MAP[actual],
            "llm_class": LABEL_MAP[predicted],
            "count": count,
            "within_reference_class_pct": round(
                count / row_totals.loc[actual] * 100,
                2
            )
        })

transition_df = pd.DataFrame(
    transitions
).sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("\nTOP 5 LABEL DISAGREEMENT PATTERNS")
display(transition_df.head(5))

# 9. Confirm known pilot results
assert int(
    class_df["disagreements"].sum()
) == 109

assert int(
    confusion.to_numpy().sum()
) == 500

# 10. Save aggregate results privately
SAVE_DIR = (
    BASE / "private_reference/evaluation/pilot_v1"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

def save_aggregate_safely(dataframe, path):
    content = dataframe.to_csv(
        index=False,
        float_format="%.6f"
    )

    if path.exists():
        assert path.read_text(
            encoding="utf-8"
        ) == content, f"Existing file differs: {path.name}"
    else:
        path.write_text(
            content,
            encoding="utf-8"
        )

save_aggregate_safely(
    class_df,
    SAVE_DIR / "classwise_disagreement.csv"
)

save_aggregate_safely(
    transition_df,
    SAVE_DIR / "label_transitions.csv"
)

print("\nCLASS-DEPENDENT ANALYSIS COMPLETED")
print("Total:", len(df))
print("Disagreements: 109")
print("Private aggregate outputs verified.")


Mounted at /content/drive
CLASS TRANSITION MATRIX (%)
Rows: Reference Labels
Columns: LLM Predictions


,World,Sports,Business,Sci/Tech
World,90.83,5.83,3.33,0.00
Sports,4.63,94.44,0.93,0.00
Business,13.29,1.40,82.52,2.80
Sci/Tech,17.83,4.65,29.46,48.06



CLASS-WISE DISAGREEMENT


,category,total,matches,disagreements,disagreement_pct
0,World,120,109,11,9.17
1,Sports,108,102,6,5.56
2,Business,143,118,25,17.48
3,Sci/Tech,129,62,67,51.94



TOP 5 LABEL DISAGREEMENT PATTERNS


,reference_class,llm_class,count,within_reference_class_pct
0,Sci/Tech,Business,38,29.46
1,Sci/Tech,World,23,17.83
2,Business,World,19,13.29
3,World,Sports,7,5.83
4,Sci/Tech,Sports,6,4.65



CLASS-DEPENDENT ANALYSIS COMPLETED
Total: 500
Disagreements: 109
Private aggregate outputs verified.


In [2]:

import json
from pathlib import Path
from datetime import datetime, timezone

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CONFIG_DIR = BASE / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

protocol = {
    "protocol_id": "s_slte_evaluation_draft_v1",
    "research_title": (
        "Evaluating Semantic-Neighborhood Label "
        "Correction for LLM-Generated Noise "
        "in Text Classification"
    ),
    "status": "DRAFT_REQUIRES_REVIEW",

    "data_policy": {
        "clean_anchor": 2000,
        "development": 1800,
        "noisy_training_pool": 16200,
        "exploratory_pilot": 500,
        "official_test": 7600,
        "pilot_used_for": [
            "descriptive error analysis",
            "feasibility analysis"
        ],
        "pilot_not_used_for": [
            "correction threshold tuning",
            "model selection",
            "confirmatory performance claims"
        ],
        "threshold_selection": (
            "Development data only, with no "
            "overlap with clean-anchor training"
        ),
        "test_policy": (
            "Keep official test data untouched "
            "until final downstream evaluation"
        )
    },

    "decisions": [
        "retain",
        "correct",
        "abstain"
    ],

    "correction_outcomes": {
        "beneficial": "wrong_to_correct",
        "harmful": "correct_to_wrong",
        "unsuccessful": "wrong_to_other_wrong",
        "unchanged_wrong": "wrong_retained",
        "unchanged_correct": "correct_retained"
    },

    "planned_metrics": [
        "noise_detection_precision",
        "noise_detection_recall",
        "noise_detection_f1",
        "beneficial_correction_count",
        "harmful_correction_count",
        "unsuccessful_correction_count",
        "correction_precision",
        "post_correction_error_rate",
        "abstention_rate",
        "classwise_metrics",
        "downstream_test_accuracy",
        "downstream_macro_f1"
    ],

    "planned_comparisons": [
        "uncorrected_LLM_labels",
        "matched_rate_symmetric_noise",
        "S_SLTE_correction",
        "selected_baseline_methods"
    ],

    "evaluation_rules": [
        "Never expose evaluation reference labels to correction.",
        "Select thresholds using development data only.",
        "Evaluate correction benefit and harm separately.",
        "Report abstention and retained-label outcomes.",
        "Use matched conditions for noise comparisons.",
        "Report negative and inconclusive results."
    ],

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

output_path = (
    CONFIG_DIR /
    "s_slte_evaluation_protocol_draft_v1.json"
)

# Avoid silently overwriting a draft
if output_path.exists():
    with open(output_path, encoding="utf-8") as f:
        saved = json.load(f)

    new_core = dict(protocol)
    old_core = dict(saved)

    new_core.pop("created_at_utc")
    old_core.pop("created_at_utc", None)

    assert old_core == new_core, (
        "Existing draft differs. Review before changing."
    )

    print("EXISTING PROTOCOL DRAFT VERIFIED")

else:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            protocol,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("EVALUATION PROTOCOL DRAFT SAVED")

print("Status:", protocol["status"])
print("Metrics:", len(protocol["planned_metrics"]))
print("Location:", output_path)


EVALUATION PROTOCOL DRAFT SAVED
Status: DRAFT_REQUIRES_REVIEW
Metrics: 12
Location: /content/drive/MyDrive/S_SLTE_Thesis/configs/s_slte_evaluation_protocol_draft_v1.json


In [3]:

import json
from pathlib import Path
from datetime import datetime, timezone

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")
CONFIG_DIR = BASE / "configs"

# Verify existing research protocol
protocol_path = (
    CONFIG_DIR /
    "s_slte_evaluation_protocol_draft_v1.json"
)

assert protocol_path.exists(), "Protocol draft missing"

with open(protocol_path, encoding="utf-8") as f:
    protocol = json.load(f)

assert protocol["status"] == "DRAFT_REQUIRES_REVIEW"

# Define mutually exclusive evaluation outcomes
definitions = {
    "document_id": "s_slte_metrics_draft_v1",
    "status": "DRAFT_REQUIRES_REVIEW",
    "purpose": "Risk-aware label correction evaluation",

    "outcome_counts": {
        "B": "Originally wrong, corrected to reference label",
        "H": "Originally correct, changed to wrong label",
        "U": "Originally wrong, changed to another wrong label",
        "RC": "Originally correct, retained",
        "RW": "Originally wrong, retained",
        "AC": "Originally correct, abstained",
        "AW": "Originally wrong, abstained"
    },

    "decision_policy": {
        "retain": "Keep original noisy label",
        "correct": "Replace with a different predicted label",
        "abstain": "Exclude from automatically accepted training labels",
        "note": "Proposed policy, subject to supervisor review"
    },

    "metric_formulas": {
        "total_records": "B+H+U+RC+RW+AC+AW",
        "correction_precision": "B/(B+H+U)",
        "harmful_correction_share": "H/(B+H+U)",
        "abstention_rate": "(AC+AW)/N",
        "coverage": "(N-AC-AW)/N",
        "accepted_error_rate": "(RW+H+U)/(N-AC-AW)",
        "original_error_rate": "(B+U+RW+AW)/N",
        "beneficial_correction_count": "B",
        "harmful_correction_count": "H"
    },

    "evaluation_rules": [
        "Every record must belong to exactly one outcome.",
        "Correction must change the original label.",
        "Zero denominators produce undefined metrics.",
        "Report coverage alongside accepted error rate.",
        "Evaluate retain, correct and abstain separately.",
        "Never expose evaluation references to correction.",
        "Do not tune thresholds on exploratory pilot labels.",
        "Use independent held-out data for confirmatory evaluation."
    ],

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

output_path = (
    CONFIG_DIR / "s_slte_metrics_draft_v1.json"
)

# Preserve existing draft if already created
if output_path.exists():
    with open(output_path, encoding="utf-8") as f:
        saved = json.load(f)

    old_core = dict(saved)
    new_core = dict(definitions)

    old_core.pop("created_at_utc", None)
    new_core.pop("created_at_utc", None)

    assert old_core == new_core, (
        "Existing definitions differ. Review required."
    )

    print("EXISTING METRICS DRAFT VERIFIED")

else:
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(definitions, f, indent=2)

    print("S-SLTE METRICS DRAFT SAVED")

print("Status:", definitions["status"])
print("Outcomes:", len(definitions["outcome_counts"]))
print("Metrics:", len(definitions["metric_formulas"]))
print("File:", output_path)


S-SLTE METRICS DRAFT SAVED
Status: DRAFT_REQUIRES_REVIEW
Outcomes: 7
Metrics: 9
File: /content/drive/MyDrive/S_SLTE_Thesis/configs/s_slte_metrics_draft_v1.json


In [4]:

import json
import math
from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/S_SLTE_Thesis")

metrics_path = (
    BASE / "configs/s_slte_metrics_draft_v1.json"
)

with open(metrics_path, encoding="utf-8") as f:
    metrics = json.load(f)

assert metrics["status"] == "DRAFT_REQUIRES_REVIEW"
assert len(metrics["outcome_counts"]) == 7
assert len(metrics["metric_formulas"]) == 9

# Synthetic records only:
# (original_label, reference_label, decision, proposed_label)
examples = [
    (1, 0, "correct", 0),  # Beneficial
    (0, 0, "correct", 1),  # Harmful
    (1, 0, "correct", 2),  # Unsuccessful
    (0, 0, "retain", None),   # Retained correct
    (1, 0, "retain", None),   # Retained wrong
    (0, 0, "abstain", None), # Abstained correct
    (1, 0, "abstain", None) # Abstained wrong
]

def classify_outcome(original, reference, decision, proposed):
    original_correct = original == reference

    if decision == "abstain":
        assert proposed is None
        return "AC" if original_correct else "AW"

    if decision == "retain":
        assert proposed is None
        return "RC" if original_correct else "RW"

    if decision == "correct":
        assert proposed in (0, 1, 2, 3)
        assert proposed != original

        if original_correct:
            return "H"

        return "B" if proposed == reference else "U"

    raise ValueError("Unknown decision")

outcomes = [
    classify_outcome(*example)
    for example in examples
]

assert outcomes == [
    "B", "H", "U", "RC", "RW", "AC", "AW"
]

counts = Counter(outcomes)

N = len(examples)
B, H, U = (
    counts["B"],
    counts["H"],
    counts["U"]
)
RC, RW = counts["RC"], counts["RW"]
AC, AW = counts["AC"], counts["AW"]

assert sum(counts.values()) == N

def safe_div(numerator, denominator):
    return (
        numerator / denominator
        if denominator > 0
        else None
    )

corrections = B + H + U
accepted = N - AC - AW

results = {
    "correction_precision": safe_div(B, corrections),
    "harmful_correction_share": safe_div(H, corrections),
    "abstention_rate": safe_div(AC + AW, N),
    "coverage": safe_div(accepted, N),
    "accepted_error_rate": safe_div(RW + H + U, accepted),
    "original_error_rate": safe_div(B + U + RW + AW, N)
}

expected = {
    "correction_precision": 1/3,
    "harmful_correction_share": 1/3,
    "abstention_rate": 2/7,
    "coverage": 5/7,
    "accepted_error_rate": 3/5,
    "original_error_rate": 4/7
}

for name, expected_value in expected.items():
    assert math.isclose(
        results[name],
        expected_value,
        rel_tol=1e-12
    ), f"Metric mismatch: {name}"

# Undefined denominator behavior
assert safe_div(0, 0) is None

print("S-SLTE METRIC CONSISTENCY TEST PASSED")
print("------------------------------------")
print("Synthetic records:", N)
print("Unique outcomes:", len(counts))

for name, value in results.items():
    print(f"{name}: {value:.4f}")

print("Zero-denominator handling: PASSED")
print("Reference data leakage: NONE (synthetic test)")


S-SLTE METRIC CONSISTENCY TEST PASSED
------------------------------------
Synthetic records: 7
Unique outcomes: 7
correction_precision: 0.3333
harmful_correction_share: 0.3333
abstention_rate: 0.2857
coverage: 0.7143
accepted_error_rate: 0.6000
original_error_rate: 0.5714
Zero-denominator handling: PASSED
Reference data leakage: NONE (synthetic test)
